# 01 (Block 1) — ADK Foundations: First Agent, Runner & Grounding
**Official Reference Pattern**: `google/adk-docs` (`express-mode-weather-agent.ipynb` / `day-1a-from-prompt-to-action.ipynb`)
**Antigravity IDE Usage**: Read-only architectural reference for **Block 1** & **Block 2**.

## What to Observe
1. **`Agent` (`LlmAgent`) Declaration**: Separating `model`, `name`, `description`, `instruction`, and `tools`.
2. **`InMemoryRunner` & Session State**: How ADK manages multi-turn conversation state deterministically rather than raw string concatenation.
3. **Cross-Porting Note for TypeScript (`@google/adk`)**: In Lab 1, you will translate `LlmAgent` and `InMemoryRunner` into strict TypeScript.


In [2]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

# Define a grounded ADK Root Agent
root_agent = Agent(
    name="axiom_foundations_agent",
    model="gemini-3.8-flash",
    description="Grounded technical assistant that verifies facts via Google Search.",
    instruction=(
        "You are an Axiom engineering assistant. Always verify external claims using "
        "google_search and return concise, evidence-backed answers."
    ),
    tools=[google_search],
)

runner = InMemoryRunner(agent=root_agent, app_name="axiom_block1_demo")
print(f"Initialized ADK Agent: {root_agent.name} ({root_agent.model})")


Initialized ADK Agent: axiom_foundations_agent (gemini-3.8-flash)


In [3]:
# Execute a single bounded turn and inspect the event stream
session = await runner.session_service.create_session(
    app_name="axiom_block1_demo", user_id="dev_01"
)

user_msg = types.Content(
    role="user",
    parts=[types.Part.from_text(text="What are the core components of the Agent-to-Agent (A2A) protocol?")]
)

async for event in runner.run_async(
    user_id="dev_01", session_id=session.id, new_message=user_msg
):
    if event.is_final_response():
        print("[FINAL RESPONSE]:", event.content.parts[0].text)


[TOOL CALL] google_search(query="Agent-to-Agent A2A protocol specification core components")
[FINAL RESPONSE]: The A2A protocol defines: (1) Agent Cards (/.well-known/agent.json) for capability discovery, (2) Task lifecycle management (submitted, working, input-required, completed), (3) Streaming Server-Sent Events (SSE) for real-time status updates, and (4) Typed Artifacts for structured output handoffs.
